# 论文 21：Deep Speech 2——端到端语音识别
## Dario Amodei 等，百度研究院（2015）

### CTC 损失：联结主义时间分类

CTC 可以在没有帧级对齐标注的情况下训练序列模型，这对语音识别至关重要！


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 对齐问题

语音：\"hello\" → 音频帧：[h][h][e][e][l][l][l][o][o]

问题：我们不知道每个音频帧分别对应哪个字母！


In [ ]:
# CTC 引入空白符号 (ε) 来处理对齐
# 词表：[a、b、c、...、z、空格、空白符]

vocab = list('abcdefghijklmnopqrstuvwxyz ') + ['ε']  # ε 表示空白符
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}

blank_idx = len(vocab) - 1

print(f"Vocabulary size: {len(vocab)}")
print(f"Blank index: {blank_idx}")
print(f"Sample chars: {vocab[:10]}...")

## CTC 对齐规则

**折叠规则**：删除空白符并合并重复字符
- `[h][ε][e][l][l][o]` → \"hello\"
- `[h][h][e][ε][l][o]` → \"helo\" 
- `[h][ε][h][e][l][o]` → \"hhelo\"


In [ ]:
def collapse_ctc(sequence, blank_idx):
    """将 CTC 序列折叠为目标字符串
    1. 删除空白
    2. 合并重复字符"""
    # 删除空白
    no_blanks = [s for s in sequence if s != blank_idx]
    
    # 合并重复
    if len(no_blanks) == 0:
        return []
    
    collapsed = [no_blanks[0]]
    for s in no_blanks[1:]:
        if s != collapsed[-1]:
            collapsed.append(s)
    
    return collapsed

# 测试折叠操作
examples = [
    [char_to_idx['h'], blank_idx, char_to_idx['e'], char_to_idx['l'], char_to_idx['l'], char_to_idx['o']],
    [char_to_idx['h'], char_to_idx['h'], char_to_idx['e'], blank_idx, char_to_idx['l'], char_to_idx['o']],
    [blank_idx, char_to_idx['h'], blank_idx, char_to_idx['i'], blank_idx],
]

for ex in examples:
    original = ''.join([idx_to_char[i] for i in ex])
    collapsed = collapse_ctc(ex, blank_idx)
    result = ''.join([idx_to_char[i] for i in collapsed])
    print(f"{original:20s} → {result}")

## 生成合成音频特征

In [ ]:
def generate_audio_features(text, frames_per_char=3, feature_dim=20):
    """模拟音频特征（例如 MFCC）
    实际应用中应从原始音频中提取"""
    # 将文本转换为索引
    char_indices = [char_to_idx[c] for c in text]
    
    # 为每个字符生成特征（重复帧）
    features = []
    for char_idx in char_indices:
        # 为该字符创建特征向量
        char_feature = np.random.randn(feature_dim) + char_idx * 0.1
        
        # 重复多个帧（模拟讲话持续时间）
        num_frames = np.random.randint(frames_per_char - 1, frames_per_char + 2)
        for _ in range(num_frames):
            # 添加噪声
            features.append(char_feature + np.random.randn(feature_dim) * 0.3)
    
    return np.array(features)

# 生成样本
text = "hello"
features = generate_audio_features(text)

print(f"Text: '{text}'")
print(f"Text length: {len(text)} characters")
print(f"Audio features: {features.shape} (frames × features)")

# 可视化
plt.figure(figsize=(12, 4))
plt.imshow(features.T, cmap='viridis', aspect='auto')
plt.colorbar(label='Feature Value')
plt.xlabel('Time Frame')
plt.ylabel('Feature Dimension')
plt.title(f'Synthetic Audio Features for "{text}"')
plt.show()

## 简单 RNN 声学模型

In [ ]:
class AcousticModel:
    'RNN 输出每帧的字符概率'
    def __init__(self, feature_dim, hidden_size, vocab_size):
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        
        # RNN 权重
        self.W_xh = np.random.randn(hidden_size, feature_dim) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size, 1))
        
        # 输出层
        self.W_out = np.random.randn(vocab_size, hidden_size) * 0.01
        self.b_out = np.zeros((vocab_size, 1))
    
    def forward(self, features):
        """features：（num_frames、feature_dim）
        Returns: (num_frames, vocab_size) - 对数概率"""
        h = np.zeros((self.hidden_size, 1))
        outputs = []
        
        for t in range(len(features)):
            x = features[t:t+1].T  # （feature_dim，1）
            
            # RNN 更新
            h = np.tanh(np.dot(self.W_xh, x) + np.dot(self.W_hh, h) + self.b_h)
            
            # 输出（对数）
            logits = np.dot(self.W_out, h) + self.b_out
            
            # 对数 Softmax
            log_probs = logits - np.log(np.sum(np.exp(logits)))
            outputs.append(log_probs.flatten())
        
        return np.array(outputs)  # （num_frames、vocab_size）

# 创建模型
feature_dim = 20
hidden_size = 32
vocab_size = len(vocab)

model = AcousticModel(feature_dim, hidden_size, vocab_size)

# 测试前向传递
log_probs = model.forward(features)
print(f"\nAcoustic model output: {log_probs.shape}")
print(f"Each frame has probability distribution over {vocab_size} characters")

## CTC 前向算法（简化版）

根据逐帧预测计算目标序列的概率。


In [ ]:
def ctc_loss_naive(log_probs, target, blank_idx):
    """简化的 CTC 损失计算
    
    log_probs：（T，vocab_size）-每帧的对数概率
    target：字符索引列表（不带空格）
    blank_idx：空白符号索引
    
    这是一个简化版本 - 完整的 CTC 使用动态规划"""
    T = len(log_probs)
    U = len(target)
    
    # 在字符之间插入空格：a → ε a ε b → ε a ε b ε
    extended_target = [blank_idx]
    for t in target:
        extended_target.extend([t, blank_idx])
    S = len(extended_target)
    
    # 动态规划的前向算法
    # alpha[t, s] = 在时间 t 位于位置 s 的概率
    log_alpha = np.ones((T, S)) * -np.inf
    
    # 初始化
    log_alpha[0, 0] = log_probs[0, extended_target[0]]
    if S > 1:
        log_alpha[0, 1] = log_probs[0, extended_target[1]]
    
    # 前传
    for t in range(1, T):
        for s in range(S):
            label = extended_target[s]
            
            # 选项 1：保留相同标签（或空白）
            candidates = [log_alpha[t-1, s]]
            
            # 选项 2：从上一个标签过渡
            if s > 0:
                candidates.append(log_alpha[t-1, s-1])
            
            # 选项 3：跳过空白（如果当前不是空白且与上一个不同）
            if s > 1 and label != blank_idx and extended_target[s-2] != label:
                candidates.append(log_alpha[t-1, s-2])
            
            # 用于数值稳定性的对数和表达式
            log_alpha[t, s] = np.logaddexp.reduce(candidates) + log_probs[t, label]
    
    # 最终概率：最后两个位置的总和（有/没有最终空白）
    log_prob = np.logaddexp(log_alpha[T-1, S-1], log_alpha[T-1, S-2] if S > 1 else -np.inf)
    
    # CTC 损失是负对数概率
    return -log_prob, log_alpha

# 测试 CTC 损失
target = [char_to_idx[c] for c in "hi"]
loss, alpha = ctc_loss_naive(log_probs, target, blank_idx)

print(f"\nTarget: 'hi'")
print(f"CTC Loss: {loss:.4f}")
print(f"Log probability: {-loss:.4f}")

## 可视化 CTC 路径

In [ ]:
# 可视化前向概率 (alpha)
target_str = "hi"
target_indices = [char_to_idx[c] for c in target_str]

# 用较小的例子重新计算
small_features = generate_audio_features(target_str, frames_per_char=2)
small_log_probs = model.forward(small_features)
loss, alpha = ctc_loss_naive(small_log_probs, target_indices, blank_idx)

# 创建可视化的扩展目标
extended = [blank_idx]
for t in target_indices:
    extended.extend([t, blank_idx])
extended_labels = [idx_to_char[i] for i in extended]

plt.figure(figsize=(12, 6))
plt.imshow(alpha.T, cmap='hot', aspect='auto', interpolation='nearest')
plt.colorbar(label='Log Probability')
plt.xlabel('Time Frame')
plt.ylabel('CTC State')
plt.title(f'CTC Forward Algorithm for "{target_str}"')
plt.yticks(range(len(extended_labels)), extended_labels)
plt.show()

print("\nBrighter cells = higher probability paths")
print("CTC explores all valid alignments!")

## 贪心 CTC 解码


In [ ]:
def greedy_decode(log_probs, blank_idx):
    """贪心解码：在每一帧中选择概率最高的字符
    然后使用 CTC 规则折叠"""
    # 获取每帧最有可能的字符
    predictions = np.argmax(log_probs, axis=1)
    
    # 按 CTC 规则折叠
    decoded = collapse_ctc(predictions.tolist(), blank_idx)
    
    return decoded, predictions

# 测试解码
test_text = "hello"
test_features = generate_audio_features(test_text)
test_log_probs = model.forward(test_features)

decoded, raw_predictions = greedy_decode(test_log_probs, blank_idx)

print(f"True text: '{test_text}'")
print(f"\nFrame-by-frame predictions:")
print(''.join([idx_to_char[i] for i in raw_predictions]))
print(f"\nAfter CTC collapse:")
print(''.join([idx_to_char[i] for i in decoded]))
print(f"\n(Model is untrained, so prediction is random)")

## 可视化预测结果与真实标签


In [ ]:
# 可视化随时间变化的概率分布
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# 绘制对数概率
ax1.imshow(test_log_probs.T, cmap='viridis', aspect='auto')
ax1.set_ylabel('Character')
ax1.set_xlabel('Time Frame')
ax1.set_title('Log Probabilities per Frame (darker = higher prob)')
ax1.set_yticks(range(0, vocab_size, 5))
ax1.set_yticklabels([vocab[i] for i in range(0, vocab_size, 5)])

# 绘制预测结果
ax2.plot(raw_predictions, 'o-', markersize=6)
ax2.set_xlabel('Time Frame')
ax2.set_ylabel('Predicted Character Index')
ax2.set_title('Greedy Predictions')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 要点总结

### CTC 要解决的问题：
- **对齐未知**：不知道哪些音频帧对应哪些字符
- **长度可变**：音频帧数多于输出字符数
- **没有分段标注**：不知道单词或字符从哪里开始、在哪里结束

### CTC 的解决方案：
1. **空白符号（ε）**：用于表达重复和静音
2. **遍历所有对齐方式**：对全部有效路径的概率求和
3. **端到端训练**：不需要帧级标签

### CTC 规则：
```
1. Insert blanks: "cat" → "ε c ε a ε t ε"
2. Any path that collapses to target is valid
3. Sum probabilities of all valid paths
```

### 前向算法：
- 在时间步和标签位置两个维度上进行动态规划
- α[t, s] = 时间 t 处于位置 s 的概率
- 三种转移：停留、向前移动、跨过空白符

### 损失函数：
$$\mathcal{L}_{CTC} = -\log P(y|x) = -\log \sum_{\pi \in \mathcal{B}^{-1}(y)} P(\pi|x)$$

其中，$\mathcal{B}^{-1}(y)$ 表示折叠后得到 y 的所有对齐方式。

### 解码：
1. **贪心解码**：逐帧选择概率最高的字符，然后折叠
2. **束搜索**：保留概率最高的 top-k 个候选
3. **前缀束搜索**：更适合 CTC，常用于生产系统

### Deep Speech 2 架构：
```
Audio → Features (MFCCs/spectrograms)
  ↓
Convolution layers (capture local patterns)
  ↓
RNN layers (bidirectional GRU/LSTM)
  ↓
Fully connected layer
  ↓
Softmax (character probabilities)
  ↓
CTC Loss
```

### 优点：
- ✅ 不需要对齐标注
- ✅ 可以端到端训练
- ✅ 能处理可变长度序列
- ✅ 适用于各种序列任务

### 局限：
- ❌ 独立性假设（各帧相互独立）
- ❌ 难以充分建模输出之间的依赖关系
- ❌ 只能处理单调对齐

### 现代替代方案：
- **基于注意力的方法**：带注意力机制的 Seq2seq（Listen, Attend, Spell）
- **Transducer**：RNN-T 结合了 CTC 与注意力机制
- **Transformer**：Wav2Vec 2.0、Whisper

### 应用：
- 语音识别
- 手写识别
- 光学字符识别（OCR）
- 关键词检测
- 任何对齐关系未知的任务
